# Controlling LLM Generation

In [1]:
from rich import print

LLMs fall under the `text-generation` task in the Transformers library. However, when instruct-tuned, they can perform any NLP task they have trained to do.

In [2]:
from transformers import pipeline

checkpoint = "Qwen/Qwen2.5-0.5B-Instruct"
generator = pipeline(task="text-generation", model=checkpoint)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## `GenerationConfig`

LLM [.generate()](https://huggingface.co/docs/transformers/v5.5.0/en/main_classes/text_generation#transformers.GenerationMixin.generate) method auto-regressively generates the next sequence given some initial `prompt`. Generation can be controlled:

See: [Common Options](https://huggingface.co/docs/transformers/v5.5.0/en/llm_tutorial#common-options):

- **`max_new_tokens` (`int`)**: Controls the maximum number of tokens generated. Make sure to set it explicitly—defaults are usually small.
- **`repetition_penalty` (`float`)**: Set to greater than 1.0 to discourage the model from repeating itself.
- **`temperature` (`float`)**: Influences randomness of generation. High values (`>0.8`) make outputs more creative; low values (`<0.4`) make them more focused and predictable.
- **`num_return_sequences` (`int`)**: Determines how many independent completions to generate for the given prompt.
- **`num_beams` (`int`)**: Activates beam search when set above 1. It is best suited for input-grounded tasks, like describing an image or speech recognition. See [this guide](https://huggingface.co/docs/transformers/v5.5.0/en/generation_strategies).

Difference between: `num_return_sequences` and `num_beams`: The latter  controls how many parallel paths the model evaluates internally, whereas the former, dictates the number of completely finished, independent text sequences handed back to you at the end of the process.

Let's see the output of `num_beams=3`:

In [3]:
# Note: __call__ and .generate() are the same
output = generator(
    "In this course, we will teach you how to",
    max_new_tokens=30,
    num_beams=3,
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [4]:
print(output[0]['generated_text'])

In this course, we will teach you how to use the Python programming language. We will cover the basics of Python, 
including syntax, data types, control structures, and functions. We will also cover

Now let's return more outputs with `num_return_sequences=5`, and loosen the model by raising it's `temperature` to get more random / creative outputs:

In [5]:
output = generator(
    "In this course, we will teach you how to",
    max_new_tokens=30,
    num_return_sequences=5,
    temperature=1.5
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [6]:
print(output)

[
    {
        'generated_text': 'In this course, we will teach you how to code for the first time.\nWhat kind of software
is used in a computer?\nProgramming language and data types are also taught during this class. These topics'
    },
    {
        'generated_text': 'In this course, we will teach you how to code in JavaScript.\n\nThis lesson starts with 
what I consider to be a "fundamental skill" of JavaScript: object oriented programming. We\'ll look at'
    },
    {
        'generated_text': 'In this course, we will teach you how to build an API server that is compatible with 
Python. We will start by building a basic Flask-based application and then move on to implementing the necessary 
functions needed'
    },
    {
        'generated_text': 'In this course, we will teach you how to make your own homemade soap.\nBefore you get 
into making your soap recipe for this class, be sure that you have the ingredients you need and know what'
    },
    {
        'generated_text': 'In this course, we will teach you how to design a user-friendly website from scratch. 
Our journey starts by creating the basic HTML and CSS files. Then we build up to building a simple webpage that'
    }
]

Notice the difference?

## LLM for Specific Tasks



Let's demonstrate that an LLM can do a specific task when prompted properly.

In [7]:
emotion_classifier_prompt = """
Act as a text classification model.
Analyze the text provided and assign it to exactly ONE category from the list below.

CATEGORIES: {categories}

TEXT: "{text}"

OUTPUT FORMAT:
Return only a JSON object:
{{
  "category": "string",
  "confidence_score": float
}}

```json
{{
"""

In [8]:
prompt = emotion_classifier_prompt.format(
    categories=['sad', 'happy', 'depressed'],
    text="I am feeling very joyful today!"
)
print(prompt)

Act as a text classification model. 
Analyze the text provided and assign it to exactly ONE category from the list below.

CATEGORIES: ['sad', 'happy', 'depressed']

TEXT: "I am feeling very joyful today!"

OUTPUT FORMAT:
Return only a JSON object:
{
  "category": "string",
  "confidence_score": float
}

```json
{

 For close-ended tasks, we usually need to be more deterministic.

In [9]:
output = generator(
    prompt,
    max_new_tokens=30,
    temperature=0.1,  # we want deterministic output
    # eos_token_id=generator.tokenizer.convert_tokens_to_ids("<output/>"),
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [10]:
llm_output = output[0]['generated_text']
print(llm_output)

Act as a text classification model. 
Analyze the text provided and assign it to exactly ONE category from the list below.

CATEGORIES: ['sad', 'happy', 'depressed']

TEXT: "I am feeling very joyful today!"

OUTPUT FORMAT:
Return only a JSON object:
{
  "category": "string",
  "confidence_score": float
}

```json
{
  "category": "happy",
  "confidence_score": 1.0
}
```

Explanation: The output should be in the format specified

We'll use regex to extract the content of the `json` block:

In [11]:
import re
import json

# Regex to find content inside ```json ... ```
match = re.search(r'```json\s+(.*?)\s+```', llm_output, re.DOTALL)

if match:
    json_str = match.group(1)
    try:
        parsed_json = json.loads(json_str)
        print("Successfully parsed JSON:")
        print(type(parsed_json), parsed_json)
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print("Extracted string was:", json_str)
else:
    print("No ```json``` block found in the output.")

Successfully parsed JSON:

<class 'dict'>
{'category': 'happy', 'confidence_score': 1.0}

Notice the problem? we have to parse the string ourselves.

## Outlines: 🗒️ Structured outputs for LLMs 🗒️


![](https://github.com/dottxt-ai/outlines/raw/main/docs/assets/images/logo-light-mode.svg#gh-light-mode-only)


## Why Outlines?

LLMs are powerful but their outputs are unpredictable. Most solutions attempt to fix bad outputs after generation using parsing, regex, or fragile code that breaks easily.

Outlines guarantees structured outputs during generation — directly from any LLM.

- **Works with any model** - Same code runs across OpenAI, Ollama, vLLM, and more
- **Simple integration** - Just pass your desired output type: `model(prompt, output_type)`
- **Guaranteed valid structure** - No more parsing headaches or broken JSON
- **Provider independence** - Switch models without changing codev


## The Outlines Philosophy

Outlines follows a simple pattern that mirrors Python's own type system. Simply specify the desired output type, and Outlines will ensure your data matches that structure exactly:

- For a yes/no response, use `Literal["Yes", "No"]`
- For numerical values, use `int`
- For complex objects, define a structure with a [Pydantic model](https://docs.pydantic.dev/latest/)

## Outlines: Core Features

|Feature|Description|Documentation|
|---|---|:-:|
|**Multiple Choices**|Constrain outputs to predefined options|[Multiple Choices Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#multiple-choices)|
|**Function Calls**|Infer structure from function signatures|[Function Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#json-schemas)|
|**JSON/Pydantic**|Generate outputs matching JSON schemas|[JSON Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#json-schemas)|
|**Regular Expressions**|Generate text following a regex pattern|[Regex Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#regex-patterns)|
|**Grammars**|Enforce complex output structures|[Grammar Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#context-free-grammars)|

In [12]:
! pip install outlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.4 MB/s eta 0:00:00


### Connect with Transformers

In [14]:
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM

model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto"),
    AutoTokenizer.from_pretrained(checkpoint)
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### Start with simple structured outputs

In [15]:
# Extract specific types
temp = model("What's the boiling point of water in Celsius?", int)
print(temp)  # 100

100

### Multiple-choice

In [16]:
from typing import Literal

# Simple classification
sentiment = model(
    "Analyze: 'This product completely changed my life!'",
    Literal["Positive", "Negative", "Neutral"]
)
print(sentiment)  # "Positive"

Positive

### Create complex structures

In [17]:
from pydantic import BaseModel
from enum import Enum

class Rating(Enum):
    poor = 1
    fair = 2
    good = 3
    excellent = 4

class ProductReview(BaseModel):
    rating: Rating
    pros: list[str]
    cons: list[str]
    summary: str

review = model(
    "Review: The XPS 13 has great battery life and a stunning display, but it runs hot and the webcam is poor quality.",
    ProductReview,
    max_new_tokens=200,
)

review = ProductReview.model_validate_json(review)
print(f"Rating: {review.rating.name}")  # "Rating: good"
print(f"Pros: {review.pros}")           # "Pros: ['great battery life', 'stunning display']"
print(f"Summary: {review.summary}")     # "Summary: Good laptop with great display but thermal issues"

Rating: fair

Pros: ["The XPS 13's battery life is impressive, making it an excellent choice for those who need to travel or have
limited power sources,"]

Summary: A mid-range laptop that excels in battery life and performance, but struggles with overheating and poor 
camera quality.